# AEGIS — Day 1 | Real-Time Supply Chain Intelligence

**Google Data Engineer / DAS-focused project** · Google Colab + GCP free trial

**What we will prove today:** Python creates a real sales event → Pub/Sub carries it → BigQuery export subscription ingests it → SQL validates and deduplicates it → stock-risk metrics update.

**Cost warning:** Check GCP Billing, create a budget alert, and keep the dataset and event volume small. BigQuery export subscriptions may incur usage charges. No Dataflow or Vertex AI deployment in Day 1.

**Do not share credentials or any internal HCLTech data. Every dataset used here is synthetic.**

## 0. GCP console checklist (do this before cloud commands)

1. Open [GCP Billing](https://console.cloud.google.com/billing): confirm free trial credits, expiry and billing project. Set a small budget alert (alerts do not stop charging).
2. Open [Create a GCP project](https://console.cloud.google.com/projectcreate): create `aegis-supply-chain-ai`; copy the **actual project ID**.
3. Open [API Library](https://console.cloud.google.com/apis/library): select the new project and enable **BigQuery API** and **Cloud Pub/Sub API**. Dataflow, Cloud Storage and Vertex AI can wait.
4. Keep [BigQuery Studio](https://console.cloud.google.com/bigquery) and [Pub/Sub](https://console.cloud.google.com/cloudpubsub/topic/list) open in other tabs.
5. Download the **AEGIS_GCP_RealTime_Project.zip** project package, then use the first cell below to upload it.

## 1. Upload the repository ZIP into Colab

Use the file picker to upload the project ZIP. The next cell unzips it, changes into the working folder and confirms that the project files are there.

In [ ]:
from google.colab import files
import os, zipfile
uploaded = files.upload()  # select AEGIS_GCP_RealTime_Project.zip
archive_name = next(name for name in uploaded if name.lower().endswith('.zip'))
with zipfile.ZipFile(archive_name) as archive:
    archive.extractall('/content')
os.chdir('/content/AEGIS_GCP_RealTime_Portfolio')
print('Working directory:', os.getcwd())
print('README found:', os.path.exists('README.md'))

## 2. Install required packages

We install the lightweight local dependencies and GCP SDK clients. **Apache Beam, neural embeddings, and Vertex AI calls are optional later**, not needed for the first streaming test.

In [ ]:
%pip -q install -r requirements.txt -r requirements-gcp.txt

## 3. Authenticate to your own GCP project

The login dialog uses your Google account. **Never paste a token here or send a screenshot showing secrets.**

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('GCP authentication complete.')

## 4. Enter your project ID (not display name)

Find it at the top of [Cloud Console](https://console.cloud.google.com/). For example, your project *name* may be `aegis-supply-chain-ai` but the ID might have an added suffix.

In [ ]:
PROJECT_ID = 'REPLACE_WITH_ACTUAL_PROJECT_ID'  # edit this line only
assert PROJECT_ID != 'REPLACE_WITH_ACTUAL_PROJECT_ID', 'Paste your actual GCP project ID.'
print('Using project:', PROJECT_ID)

## 5. Verify the project works with BigQuery

In [ ]:
from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)
result = list(client.query('SELECT 1 AS connection_ok', location='US').result())
print('BigQuery connection result:', result[0].connection_ok)

## 6. Create synthetic product, supplier, warehouse, inventory and demand data

The `synthetic.py` generator writes six CSV datasets. This is **fully fabricated portfolio data**, not enterprise information.

In [ ]:
from aegis.synthetic import generate
frames = generate()
for name, df in frames.items():
    print(f'{name:20s}: {len(df):>4} rows')
display(frames['products'])
display(frames['inventory'].head())

## 7. Run offline tests before using any cloud services

Every test passing means our synthetic keys, event validation, scenario math and basic data contracts are consistent.

In [ ]:
!python -m pytest -q

## 8. Demonstrate our distinctive Supplier Delay Decision Replay

In [ ]:
import pandas as pd
from aegis.risk import decision_replay
scenarios = pd.DataFrame(decision_replay())
display(scenarios[['delay_days','first_stockout_day','unfulfilled_units','potential_revenue_exposure_inr','incremental_exposure_inr']])
ax = scenarios.plot(x='delay_days', y='potential_revenue_exposure_inr', kind='bar',
                    title='Hypothetical revenue exposure vs supplier delay',legend=False)
ax.set_ylabel('INR (scenario estimate)')

## 9. Create BigQuery warehouse tables and Pub/Sub topic

This runs the included setup script. It creates the dataset, raw event table, six reference tables, SQL views and Pub/Sub topic. This **does not** yet create a BigQuery subscription.

**If you get permission errors, send the exact error text (without sensitive credentials).**

In [ ]:
!python -m scripts.setup_gcp --project-id {PROJECT_ID}

## 10. IMPORTANT — manually connect Pub/Sub to BigQuery in Cloud Console

1. Go to [Pub/Sub Subscriptions](https://console.cloud.google.com/cloudpubsub/subscription/list).
2. Click **Create subscription**.
3. Subscription ID: `aegis-to-bigquery`.
4. Topic: `aegis-sales-events` in your project.
5. Delivery type: **Write to BigQuery**.
6. Destination: project → dataset `aegis_warehouse` → table `raw_sales_events`.
7. Schema configuration: **Don't use a schema**. Leave **Write metadata** off.
8. Click **Create**.

If the Console complains about permissions, the Pub/Sub service agent requires `BigQuery Data Editor` **on the destination dataset**. See [Google's official guide](https://cloud.google.com/pubsub/docs/create-bigquery-subscription). Share the exact error so we can apply the minimum permission needed.

**Create this subscription BEFORE publishing the demo events**, otherwise old topic messages may not get delivered automatically. Pause here until Console shows the subscription.

## 11. Publish 20 real events and intentionally repeat the first event

A message is sent to managed Pub/Sub. The repeated `event_id` tests whether our SQL view avoids double-counting logical sales.

In [ ]:
!python -m scripts.publish_events --project-id {PROJECT_ID} --count 20 --replay-first

## 12. Query BigQuery to verify the live pipeline

Allow Pub/Sub some time to deliver events. Rerun this cell if the first result is still zero. You should see about 21 raw messages and 20 unique validated events if you started from an empty topic/subscription and all messages were delivered.

In [ ]:
!python -m scripts.run_queries --project-id {PROJECT_ID}

## 13. Show data in BigQuery Studio

Open [BigQuery Studio](https://console.cloud.google.com/bigquery) → select your project → `aegis_warehouse` → `v_stock_risk` → Preview. Or run:

```sql
SELECT * FROM `YOUR_PROJECT_ID.aegis_warehouse.v_stock_risk`
ORDER BY estimated_days_cover ASC LIMIT 12;
```

Take proof screenshots of **the Pub/Sub topic/subscription**, **raw event count**, **dedup event count**, and **stock-risk output**. Avoid billing/account-private screenshots.

**Day 1 success:** You can trace a synthetic sales event from Python through Pub/Sub into BigQuery and understand every transformation. Now open `docs/DAY2_ADVANCED.md` for the optional ML/RAG/Dataflow steps.